# Incident Response Runbook: Credential Access - OS Credential Dumping

**Tactic:** Credential Access
**Technique:** OS Credential Dumping (T1003)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for OS Credential Dumping activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import subprocess
import os

# Import security integrations
from autobook.runbook_loader import *
from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

alert_data = {
    'alert_id': 'ALERT-001',
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Credential Access',
    'technique': 'OS Credential Dumping',
    'severity': 'CRITICAL',
    'incident_id': f"IR-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
}

print(f"Alert ID: {alert_data['alert_id']}")
print(f"Timestamp: {alert_data['timestamp']}")
print(f"Tactic: {alert_data['tactic']}")
print(f"Technique: {alert_data['technique']}")
print(f"Severity: {alert_data['severity']}")
print(f"Incident ID: {alert_data['incident_id']}")

# Query Splunk for OS credential dumping indicators
print(f"\n[ACTION] Querying Splunk for OS credential dumping indicators...")
credential_dump_queries = [
    # LSASS memory access
    'index=* (sourcetype=WinEventLog:Security OR sourcetype=Syslog) "lsass.exe" "memory access" OR "credential dump" OR "mimikatz"',
    # SAM database access
    'index=* sourcetype=WinEventLog:Security EventCode=4656 "SAM" OR "SYSTEM" "registry access" "credential theft"',
    # NTDS.dit access
    'index=* sourcetype=WinEventLog:Security "NTDS.dit" OR "Active Directory" "database dump" OR "domain credentials"',
    # Suspicious process accessing credential stores
    'index=* sourcetype=WinEventLog:Security EventCode=4688 (lsass.exe OR sam.exe OR ntdsutil.exe) "suspicious parent" OR "anomalous access"',
    # PowerShell credential dumping
    'index=* sourcetype=WinEventLog:Security "powershell" ("get-credential" OR "convertfrom-securestring" OR "credential dump")'
]

suspicious_activities = []
affected_systems = []

for query in credential_dump_queries:
    try:
        results = splunk.search(query, time_range='-24h')
        if results:
            suspicious_activities.extend(results)
            print(f"   Found {len(results)} credential dumping indicators")
        else:
            print(f"  - No results for query: {query[:50]}...")
    except Exception as e:
        print(f"   Query failed: {e}")

# Analyze with CrowdStrike for credential dumping patterns
print(f"\n[ACTION] Analyzing with CrowdStrike for credential dumping patterns...")
for activity in suspicious_activities:
    try:
        host_analysis = crowdstrike.analyze_host(activity.get('host', ''))
        if host_analysis.get('credential_dumping_detected'):
            affected_systems.append({
                'hostname': activity.get('host', ''),
                'device_id': host_analysis.get('device_id', ''),
                'dumping_indicators': host_analysis.get('indicators', []),
                'compromised_accounts': host_analysis.get('compromised_accounts', [])
            })
            print(f"   Credential dumping detected on {activity.get('host', '')}: {len(host_analysis.get('indicators', []))} indicators, {len(host_analysis.get('compromised_accounts', []))} accounts affected")
    except Exception as e:
        print(f"   Host analysis failed for {activity.get('host', '')}: {e}")

# Enrich with threat intelligence
print(f"\n[ACTION] Enriching with threat intelligence...")
for activity in suspicious_activities:
    try:
        # Search for tool hashes or process names in MISP
        if activity.get('process_hash'):
            misp_results = misp.search_iocs(activity['process_hash'], type='hash')
            if misp_results:
                activity['threat_intel'] = misp_results[:3]
                print(f"   Threat intel found for process hash {activity.get('process_hash')[:16]}...: {len(misp_results)} matches")
        elif activity.get('process_name'):
            misp_results = misp.search_iocs(activity['process_name'], type='filename')
            if misp_results:
                activity['threat_intel'] = misp_results[:3]
                print(f"   Threat intel found for process {activity.get('process_name')}: {len(misp_results)} matches")
    except Exception as e:
        print(f"   Threat intel lookup failed: {e}")

# Create IRIS case
print(f"\n[ACTION] Creating IRIS incident case...")
try:
    iris_case = iris.create_case(
        title=f"Credential Access - OS Credential Dumping Incident {alert_data['incident_id']}",
        description=f"Detected OS credential dumping activities on {len(affected_systems)} systems with {len(suspicious_activities)} suspicious indicators.",
        severity='critical',
        tags=['credential-access', 'os-credential-dumping', 't1003']
    )
    alert_data['iris_case_id'] = iris_case.get('case_id')
    print(f"   IRIS case created: {iris_case.get('case_id')}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(suspicious_activities)} suspicious credential dumping activities detected")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities")
print(f"  - IRIS case created: {alert_data.get('iris_case_id', 'none')}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

isolated_hosts = []
blocked_entities = []
disabled_accounts = []

# Isolate affected systems immediately
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        isolation_result = crowdstrike.isolate_host(system['device_id'])
        if isolation_result.get('status') == 'isolated':
            isolated_hosts.append(system['hostname'])
            print(f"   Isolated {system['hostname']}")
        else:
            print(f"   Failed to isolate {system['hostname']}: {isolation_result.get('message', 'unknown error')}")
    except Exception as e:
        print(f"   Isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Disable compromised accounts
print(f"\n[ACTION] Disabling compromised accounts...")
for system in affected_systems:
    try:
        for account in system.get('compromised_accounts', []):
            account_disable = shuffle.disable_account(account)
            if account_disable.get('status') == 'disabled':
                disabled_accounts.append(account)
                print(f"   Disabled compromised account: {account}")
    except Exception as e:
        print(f"   Account disabling failed: {e}")

# Block associated IPs and domains
print(f"\n[ACTION] Blocking associated IPs and domains...")
for activity in suspicious_activities:
    try:
        if activity.get('source_ip'):
            block_result = shuffle.block_ip(activity['source_ip'])
            if block_result.get('status') == 'blocked':
                blocked_entities.append(f"IP:{activity['source_ip']}")
                print(f"   Blocked source IP: {activity['source_ip']}")

        if activity.get('destination_ip'):
            block_result = shuffle.block_ip(activity['destination_ip'])
            if block_result.get('status') == 'blocked':
                blocked_entities.append(f"IP:{activity['destination_ip']}")
                print(f"   Blocked destination IP: {activity['destination_ip']}")

        if activity.get('domain'):
            domain_block = shuffle.block_domain(activity['domain'])
            if domain_block.get('status') == 'blocked':
                blocked_entities.append(f"Domain:{activity['domain']}")
                print(f"   Blocked domain: {activity['domain']}")
    except Exception as e:
        print(f"   Entity blocking failed: {e}")

# Terminate credential dumping processes
print(f"\n[ACTION] Terminating credential dumping processes...")
terminated_processes = []
for activity in suspicious_activities:
    try:
        if activity.get('process_id') and activity.get('host'):
            termination_result = crowdstrike.terminate_process(
                activity['host'],
                activity['process_id']
            )
            if termination_result.get('status') == 'terminated':
                terminated_processes.append(f"{activity['host']}:{activity['process_id']}")
                print(f"   Terminated credential dumping process {activity['process_id']} on {activity['host']}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Quarantine credential dumping tools
print(f"\n[ACTION] Quarantining credential dumping tools...")
quarantined_tools = []
for activity in suspicious_activities:
    try:
        if activity.get('tool_path') and activity.get('host'):
            quarantine_result = crowdstrike.quarantine_file(
                activity['host'],
                activity['tool_path']
            )
            if quarantine_result.get('status') == 'quarantined':
                quarantined_tools.append(activity['tool_path'])
                print(f"   Quarantined credential dumping tool: {activity['tool_path']}")
    except Exception as e:
        print(f"   Tool quarantine failed for {activity.get('tool_path', 'unknown')}: {e}")

# Enable enhanced monitoring for credential access
print(f"\n[ACTION] Enabling enhanced monitoring for credential access...")
try:
    monitoring_result = shuffle.enable_enhanced_monitoring(
        targets=[s['hostname'] for s in affected_systems],
        rules=['credential_dump_detection', 'lsass_access', 'sam_access', 'suspicious_credential_processes']
    )
    if monitoring_result.get('status') == 'enabled':
        print(f"   Enhanced credential monitoring enabled for {len(affected_systems)} systems")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(disabled_accounts)} compromised accounts disabled")
print(f"  - {len(blocked_entities)} entities blocked")
print(f"  - {len(terminated_processes)} credential dumping processes terminated")
print(f"  - {len(quarantined_tools)} credential dumping tools quarantined")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_tools = []
reset_credentials = []
cleaned_systems = []

# Remove credential dumping tools
print(f"\n[ACTION] Removing credential dumping tools...")
for activity in suspicious_activities:
    try:
        if activity.get('tool_path') and activity.get('host'):
            removal_result = crowdstrike.remove_file(
                activity['host'],
                activity['tool_path']
            )
            if removal_result.get('status') == 'removed':
                removed_tools.append(activity['tool_path'])
                print(f"   Removed credential dumping tool: {activity['tool_path']}")
    except Exception as e:
        print(f"   Tool removal failed for {activity.get('tool_path', 'unknown')}: {e}")

# Clean dumped credential artifacts
print(f"\n[ACTION] Cleaning dumped credential artifacts...")
cleaned_artifacts = []
for activity in suspicious_activities:
    try:
        if activity.get('dump_file_path') and activity.get('host'):
            cleanup_result = crowdstrike.remove_file(
                activity['host'],
                activity['dump_file_path']
            )
            if cleanup_result.get('status') == 'removed':
                cleaned_artifacts.append(activity['dump_file_path'])
                print(f"   Cleaned dumped credentials file: {activity['dump_file_path']}")
    except Exception as e:
        print(f"   Artifact cleanup failed for {activity.get('dump_file_path', 'unknown')}: {e}")

# Reset all compromised credentials
print(f"\n[ACTION] Resetting compromised credentials...")
for system in affected_systems:
    try:
        for account in system.get('compromised_accounts', []):
            reset_result = shuffle.reset_password(account)
            if reset_result.get('status') == 'reset':
                reset_credentials.append(account)
                print(f"   Reset password for compromised account: {account}")
    except Exception as e:
        print(f"   Password reset failed for account in {system.get('hostname', 'unknown')}: {e}")

# Clean memory and caches
print(f"\n[ACTION] Cleaning memory and credential caches...")
memory_cleaned = []
for system in affected_systems:
    try:
        memory_result = crowdstrike.clear_credential_cache(system['device_id'])
        if memory_result.get('status') == 'cleared':
            memory_cleaned.append(system['hostname'])
            print(f"   Cleared credential cache on {system['hostname']}")
    except Exception as e:
        print(f"   Memory cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Remove persistence mechanisms
print(f"\n[ACTION] Removing persistence mechanisms...")
removed_persistence = []
for system in affected_systems:
    try:
        persistence_result = crowdstrike.remove_credential_persistence(system['device_id'])
        if persistence_result.get('status') == 'removed':
            removed_persistence.append(system['hostname'])
            print(f"   Removed credential persistence mechanisms from {system['hostname']}")
    except Exception as e:
        print(f"   Persistence removal failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        verify_result = crowdstrike.verify_credential_dump_removal(system['device_id'])
        verification_results.append({
            'system': system['hostname'],
            'status': 'clean' if verify_result.get('credential_dumping_detected', True) == False else 'threats_remaining',
            'remaining_indicators': verify_result.get('remaining_indicators', 0)
        })
        status = "" if verify_result.get('credential_dumping_detected', True) == False else ""
        print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} credential dumping indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_tools)} credential dumping tools removed")
print(f"  - {len(cleaned_artifacts)} dumped credential artifacts cleaned")
print(f"  - {len(reset_credentials)} compromised credentials reset")
print(f"  - {len(memory_cleaned)} systems had credential caches cleared")
print(f"  - {len(removed_persistence)} systems had persistence mechanisms removed")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
reconnected_hosts = []
validated_credentials = []

# Restore systems from clean backups
print(f"\n[ACTION] Restoring systems from clean backups...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            restore_result = crowdstrike.restore_from_backup(
                system['device_id'],
                backup_type='clean'
            )
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored {system['hostname']} from clean backup")
            else:
                print(f"   No clean backup available for {system['hostname']}, manual restoration required")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Reconnect isolated systems
print(f"\n[ACTION] Reconnecting isolated systems...")
for system in affected_systems:
    try:
        if system['hostname'] in isolated_hosts:
            reconnect_result = crowdstrike.reconnect_host(system['device_id'])
            if reconnect_result.get('status') == 'reconnected':
                reconnected_hosts.append(system['hostname'])
                print(f"   Reconnected {system['hostname']} to network")
    except Exception as e:
        print(f"   Reconnection failed for {system.get('hostname', 'unknown')}: {e}")

# Validate credential integrity
print(f"\n[ACTION] Validating credential integrity...")
for system in affected_systems:
    try:
        credential_validation = crowdstrike.validate_credentials(system['device_id'])
        if credential_validation.get('status') == 'validated':
            validated_credentials.append(system['hostname'])
            print(f"   Credential integrity validated for {system['hostname']}")
        else:
            issues = credential_validation.get('issues', [])
            print(f"   Credential issues found for {system['hostname']}: {len(issues)} issues")
    except Exception as e:
        print(f"   Credential validation failed for {system.get('hostname', 'unknown')}: {e}")

# Restore legitimate accounts
print(f"\n[ACTION] Restoring legitimate accounts...")
restored_accounts = []
for account in disabled_accounts:
    try:
        # Verify account is legitimate before restoration
        account_check = shuffle.verify_account_legitimacy(account)
        if account_check.get('legitimate', False):
            restore_result = shuffle.restore_account(account)
            if restore_result.get('status') == 'restored':
                restored_accounts.append(account)
                print(f"   Restored legitimate account: {account}")
        else:
            print(f"   Account {account} flagged as compromised, keeping disabled")
    except Exception as e:
        print(f"   Account restoration failed for {account}: {e}")

# Implement credential rotation policy
print(f"\n[ACTION] Implementing enhanced credential policies...")
policy_updates = []
try:
    # Update password policy
    password_policy = shuffle.update_password_policy(
        rules={
            'minimum_length': 16,
            'complexity_requirements': True,
            'rotation_frequency': 90,
            'prevent_reuse': True
        }
    )
    if password_policy.get('status') == 'updated':
        policy_updates.append('password_policy')
        print(f"   Updated password policy for enhanced security")

    # Implement multi-factor authentication
    mfa_policy = shuffle.enforce_mfa_policy(
        accounts=[acc for acc in disabled_accounts if acc in restored_accounts],
        required=True
    )
    if mfa_policy.get('status') == 'enforced':
        policy_updates.append('mfa_policy')
        print(f"   Enforced MFA policy for restored accounts")
except Exception as e:
    print(f"   Policy update failed: {e}")

# Set up continuous credential monitoring
print(f"\n[ACTION] Setting up continuous credential monitoring...")
monitoring_systems = []
for system in affected_systems:
    try:
        credential_monitoring = crowdstrike.setup_credential_monitoring(
            system['device_id'],
            rules=['anomalous_credential_access', 'credential_dump_attempts', 'privilege_escalation']
        )
        if credential_monitoring.get('status') == 'monitoring':
            monitoring_systems.append(system['hostname'])
            print(f"   Continuous credential monitoring enabled for {system['hostname']}")
    except Exception as e:
        print(f"   Credential monitoring setup failed for {system.get('hostname', 'unknown')}: {e}")

# Update security policies
print(f"\n[ACTION] Updating security policies to prevent credential dumping...")
try:
    # Enhance process monitoring for credential access
    process_policy = crowdstrike.update_security_policy(
        policy_type='process_monitoring',
        rules={
            'lsass_protection': True,
            'sam_protection': True,
            'credential_process_monitoring': True,
            'memory_access_control': True
        }
    )
    if process_policy.get('status') == 'updated':
        policy_updates.append('process_monitoring')
        print(f"   Updated process monitoring policy for credential protection")

    # Update behavioral analytics
    behavior_policy = crowdstrike.update_security_policy(
        policy_type='behavioral_analytics',
        rules={
            'credential_anomaly_detection': True,
            'privilege_escalation_detection': True,
            'suspicious_memory_access': True
        }
    )
    if behavior_policy.get('status') == 'updated':
        policy_updates.append('behavioral_analytics')
        print(f"   Updated behavioral analytics policy")
except Exception as e:
    print(f"   Policy update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(reconnected_hosts)} systems reconnected")
print(f"  - {len(validated_credentials)} systems had credentials validated")
print(f"  - {len(restored_accounts)} legitimate accounts restored")
print(f"  - {len(monitoring_systems)} systems under continuous credential monitoring")
print(f"  - {len(policy_updates)} security policies updated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Document incident details
incident_summary = {
    'incident_id': alert_data.get('incident_id', 'unknown'),
    'technique': 'Credential Access - OS Credential Dumping',
    'detection_time': alert_data.get('timestamp', 'unknown'),
    'affected_systems': len(affected_systems),
    'credential_dumping_indicators': len(suspicious_activities),
    'compromised_accounts': sum(len(s.get('compromised_accounts', [])) for s in affected_systems),
    'containment_actions': len(isolated_hosts) + len(blocked_entities) + len(disabled_accounts),
    'eradication_actions': len(removed_tools) + len(cleaned_artifacts) + len(reset_credentials),
    'recovery_actions': len(restored_systems) + len(reconnected_hosts) + len(validated_credentials),
    'timeline': {
        'detection': alert_data.get('timestamp', 'unknown'),
        'containment': 'completed',
        'eradication': 'completed',
        'recovery': 'completed',
        'closure': 'now'
    }
}

# Create comprehensive incident report
print(f"\n[ACTION] Creating comprehensive incident report...")
try:
    iris_report = iris.create_incident_report(
        incident_id=alert_data.get('incident_id', 'unknown'),
        title=f"Credential Access - OS Credential Dumping Incident {alert_data.get('incident_id', 'unknown')}",
        description=f"Detected and responded to OS credential dumping activities on {len(affected_systems)} systems. {len(suspicious_activities)} suspicious indicators were identified and {sum(len(s.get('compromised_accounts', [])) for s in affected_systems)} accounts were compromised.",
        severity='critical',
        status='closed',
        details={
            'technique': 'T1003 - OS Credential Dumping',
            'indicators': [activity.get('description', 'Unknown credential dumping activity') for activity in suspicious_activities[:10]],
            'affected_assets': [s['hostname'] for s in affected_systems],
            'compromised_accounts': [account for s in affected_systems for account in s.get('compromised_accounts', [])],
            'response_actions': {
                'detection': f"Splunk queries identified {len(suspicious_activities)} credential dumping indicators",
                'containment': f"Isolated {len(isolated_hosts)} hosts, blocked {len(blocked_entities)} entities, disabled {len(disabled_accounts)} accounts",
                'eradication': f"Removed {len(removed_tools)} tools, cleaned {len(cleaned_artifacts)} artifacts, reset {len(reset_credentials)} credentials",
                'recovery': f"Restored {len(restored_systems)} systems, reconnected {len(reconnected_hosts)} hosts, validated {len(validated_credentials)} systems"
            },
            'threat_intelligence': {
                'misp_enrichment': len([a for a in suspicious_activities if a.get('threat_intel')]),
                'high_confidence_indicators': len([a for a in suspicious_activities if a.get('threat_intel') and len(a['threat_intel']) > 0])
            }
        }
    )
    print(f"   Incident report created in IRIS: {iris_report.get('report_id', 'unknown')}")
except Exception as e:
    print(f"   Report creation failed: {e}")

# Share indicators with threat intelligence community
print(f"\n[ACTION] Sharing indicators with threat intelligence community...")
try:
    shared_iocs = []
    for activity in suspicious_activities:
        if activity.get('threat_intel') and len(activity['threat_intel']) > 0:
            # Share tool hashes, process names, and associated IPs
            ioc_attributes = []
            if activity.get('process_hash'):
                ioc_attributes.append({
                    'type': 'sha256',
                    'value': activity['process_hash'],
                    'comment': f'Credential dumping tool from incident {alert_data.get("incident_id", "unknown")}'
                })
            if activity.get('tool_hash'):
                ioc_attributes.append({
                    'type': 'sha256',
                    'value': activity['tool_hash'],
                    'comment': f'Credential dumping tool hash from incident {alert_data.get("incident_id", "unknown")}'
                })
            if activity.get('source_ip'):
                ioc_attributes.append({
                    'type': 'ip-src',
                    'value': activity['source_ip'],
                    'comment': f'Source IP associated with credential dumping activity'
                })
            if activity.get('process_name'):
                ioc_attributes.append({
                    'type': 'filename',
                    'value': activity['process_name'],
                    'comment': f'Credential dumping process name'
                })

            if ioc_attributes:
                misp_event = misp.create_event(
                    title=f"OS Credential Dumping Detection: {activity.get('description', 'Unknown')}",
                    description=f"OS credential dumping technique detected during incident response. Activity: {activity.get('description', 'Unknown')}",
                    attributes=ioc_attributes
                )
                if misp_event.get('status') == 'created':
                    shared_iocs.extend([attr['value'] for attr in ioc_attributes])
                    print(f"   Shared {len(ioc_attributes)} IOCs for credential dumping activity")

    print(f"   Shared {len(shared_iocs)} indicators with threat intelligence community")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

# Generate lessons learned
print(f"\n[ACTION] Generating lessons learned...")
lessons_learned = {
    'incident_type': 'Credential Access - OS Credential Dumping',
    'key_findings': [
        f"Credential dumping indicators detected: {len(suspicious_activities)} across {len(affected_systems)} systems",
        f"Compromised accounts: {sum(len(s.get('compromised_accounts', [])) for s in affected_systems)} total",
        f"Most common dumping method: {max(set([a.get('dumping_method', 'unknown') for a in suspicious_activities]), key=[a.get('dumping_method', 'unknown') for a in suspicious_activities].count) if suspicious_activities else 'none'}",
        f"Threat intelligence enrichment: {len([a for a in suspicious_activities if a.get('threat_intel')])} activities enriched",
        f"Containment effectiveness: {len(isolated_hosts)}/{len(affected_systems)} systems isolated within detection window"
    ],
    'improvements_needed': [
        'Enhance LSASS and SAM protection mechanisms',
        'Implement real-time credential access monitoring',
        'Strengthen memory protection and access controls',
        'Improve automated credential compromise detection'
    ],
    'preventive_measures': [
        'Deploy advanced endpoint credential protection',
        'Implement behavioral analytics for credential access patterns',
        'Regular security policy updates for emerging credential dumping techniques',
        'Enhanced threat intelligence integration for known credential dumping tools'
    ]
}

try:
    iris.add_lessons_learned(
        incident_id=alert_data.get('incident_id', 'unknown'),
        lessons=lessons_learned
    )
    print(f"   Lessons learned documented in IRIS")
except Exception as e:
    print(f"   Lessons learned documentation failed: {e}")

# Final status update
print(f"\n Incident {alert_data.get('incident_id', 'unknown')} successfully resolved:")
print(f"  - Technique: Credential Access - OS Credential Dumping (T1003)")
print(f"  - Status: Closed")
print(f"  - Systems Recovered: {len(restored_systems) + len(reconnected_hosts)}")
print(f"  - Accounts Secured: {len(reset_credentials)} passwords reset, {len(restored_accounts)} accounts restored")
print(f"  - Threat Indicators Shared: {len(shared_iocs) if 'shared_iocs' in locals() else 0}")
print(f"  - Incident Report Generated: Yes")
print(f"  - Lessons Learned Documented: Yes")

print(f"\n{'=' * 60}")
print("INCIDENT RESPONSE COMPLETE")
print(f"{'=' * 60}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
